In [95]:
import os
import sys
sys.path.append('/workspace/Retinal-vessels-segmentation/')

In [96]:
import numpy as np
from PIL import Image

In [97]:
import torch
import torch.nn as nn
from utils import *
import matplotlib.pyplot as plt
import cv2
from io import BytesIO
import torch.nn.functional as F
from transforms import get_test_patch_transforms
from sklearn.metrics import f1_score, recall_score

In [98]:
def preprocessing_img(path):
    mean_=73.00342685729963
    std_=54.45611922239714
    if isinstance(path,str):
        img=np.array(Image.open(path).convert('RGB'))
    else:
        img=path

    clahe = cv2.createCLAHE(clipLimit=5.0, tileGridSize=(8,8))

    gray=convert_gray(img)
    gray=(gray-mean_)/std_
    gray=((gray-np.min(gray))/(np.max(gray)-np.min(gray)))*255
    
    gray=clahe.apply(np.array(gray,dtype=np.uint8))
    return unsharp_mask(gray)

In [99]:
def infer_model(model: nn.Module, image: np.ndarray| str, device: torch.device='cuda',output_path: str|None=None,preprocessing_func=None) -> np.ndarray:
    model = model.to(device)
    model.eval()
    with torch.inference_mode():
        if isinstance(image, str):
            image = np.array(Image.open(image))
        preprocessed_image = preprocessing_func(image)
        img_tensor = get_test_patch_transforms()(image=preprocessed_image)['image'].to(device)
        _, original_h, original_w = img_tensor.shape

        
        img_tensor = mirror_padding_v2(img_tensor).unsqueeze(0)
        B, C, H, W = img_tensor.shape
        num_patch = ((H-64)//32+1, (W-64)//8+1)
        image_patches, tmp_stride = extract_patches_with_target_count(img_tensor, 64, num_patch)
        if len(image_patches.shape) > 4:
            image_patches = image_patches.flatten(0, 1)
        chunk_size = max(image_patches.shape[0] // 128, 1)
        chunk_image = torch.chunk(image_patches, chunk_size, 0)

        out_sample = []
        for c_image in chunk_image:
            with torch.inference_mode():
                prob = model(c_image)
            out_sample.append(prob)
        prob = torch.cat(out_sample, 0)
        prob = prob.view(B, -1, 1, 64, 64)
        prob = reverse_to_original_image(prob, (H, W), 64, tmp_stride).squeeze()[:original_h, :original_w]

        # Threshold => binary mask (numpy)
        pred_mask = (prob >= 0.487).to(torch.uint8).detach().cpu().numpy()  # shape (h,w), 0/1
        seg_display = (pred_mask * 255).astype(np.uint8)
    if output_path is not None:
        cv2.imwrite(output_path, seg_display)   
    return seg_display

In [100]:
def save_png_to_bytes(img):
    """
    img: np.ndarray (H,W) or (H,W,3) uint8
         or PIL.Image
    return: bytes PNG
    """
    buf = BytesIO()
    if isinstance(img, np.ndarray):
        Image.fromarray(img).save(buf, format="PNG")
    else:
        img.save(buf, format="PNG")
    buf.seek(0)
    return buf.getvalue()

def draw_dashed_line(img, pt1, pt2, color, thickness=1, gap=10):
    """Draw a dashed line between two points"""
    dist = ((pt1[0] - pt2[0]) ** 2 + (pt1[1] - pt2[1]) ** 2) ** 0.5
    pts = []
    for i in np.arange(0, dist, gap):
        r = i / dist
        x = int((pt1[0] * (1 - r) + pt2[0] * r) + 0.5)
        y = int((pt1[1] * (1 - r) + pt2[1] * r) + 0.5)
        pts.append((x, y))
    
    for i in range(0, len(pts) - 1, 2):
        cv2.line(img, pts[i], pts[i + 1], color, thickness)

def create_zoom_inset(image_path, output_path,pos=(243, 419, 48, 30),resize_size=(1024,1024),scale_roi=2, line_style='solid'):
    """
    line_style options:
    - 'solid': solid lines
    - 'dashed': dashed lines
    - 'both': connects all 4 corners
    - 'diagonal': connects diagonal corners
    """
    img = cv2.imread(image_path)
    h, w = img.shape[:2]

    # 1. Xác định vị trí vùng muốn cắt (x, y, width, height)
    crop_x, crop_y, crop_w, crop_h = pos 
    
    # 2. Cắt vùng đó ra
    roi = img[crop_y:crop_y+crop_h, crop_x:crop_x+crop_w]

    # 3. Phóng to vùng đã cắt
    zoom_scale = scale_roi
    zoomed_roi = cv2.resize(roi, None, fx=zoom_scale, fy=zoom_scale, interpolation=cv2.INTER_LANCZOS4)

    # 4. Vẽ khung vàng cho vùng cắt trên ảnh gốc
    cv2.rectangle(img, (crop_x, crop_y), (crop_x+crop_w, crop_y+crop_h), (0, 255, 0), 2)

    # 5. Xác định vị trí đặt ảnh đã zoom
    zh, zw = zoomed_roi.shape[:2]
    pos_x, pos_y = w - zw, h - zh
    
    # Vẽ khung vàng cho ảnh zoom
    cv2.rectangle(zoomed_roi, (0, 0), (zw-1, zh-1), (0, 255, 0), 8)

    # 6. Đè ảnh zoom lên ảnh gốc
    img[pos_y:pos_y+zh, pos_x:pos_x+zw] = zoomed_roi

    # 7. VẼ CÁC ĐƯỜNG NỐI
    line_color = (0, 255, 255)  # Yellow (BGR)
    line_thickness = 5
    
    # Define corner points of ROI (original region)
    roi_corners = {
        'top_left': (crop_x, crop_y),
        'top_right': (crop_x + crop_w, crop_y),
        'bottom_left': (crop_x, crop_y + crop_h),
        'bottom_right': (crop_x + crop_w, crop_y + crop_h)
    }
    
    # Define corner points of zoomed inset
    zoom_corners = {
        'top_left': (pos_x, pos_y),
        'top_right': (pos_x + zw, pos_y),
        'bottom_left': (pos_x, pos_y + zh),
        'bottom_right': (pos_x + zw, pos_y + zh)
    }
    
    if line_style == 'solid':
        # Connect corresponding corners with solid lines
        cv2.line(img, roi_corners['top_right'], zoom_corners['top_left'], line_color, line_thickness)
        cv2.line(img, roi_corners['bottom_right'], zoom_corners['bottom_left'], line_color, line_thickness)
    
    elif line_style == 'dashed':
        # Connect corresponding corners with dashed lines
        draw_dashed_line(img, roi_corners['top_right'], zoom_corners['top_left'], line_color, line_thickness)
        draw_dashed_line(img, roi_corners['bottom_right'], zoom_corners['bottom_left'], line_color, line_thickness)
    
    elif line_style == 'both':
        # Connect all 4 corners
        draw_dashed_line(img, roi_corners['top_left'], zoom_corners['top_left'], line_color, line_thickness)
        draw_dashed_line(img, roi_corners['top_right'], zoom_corners['top_right'], line_color, line_thickness)
        draw_dashed_line(img, roi_corners['bottom_left'], zoom_corners['bottom_left'], line_color, line_thickness)
        draw_dashed_line(img, roi_corners['bottom_right'], zoom_corners['bottom_right'], line_color, line_thickness)
    
    elif line_style == 'diagonal':
        # Connect diagonal corners
        draw_dashed_line(img, roi_corners['top_left'], zoom_corners['bottom_right'], line_color, line_thickness)
        draw_dashed_line(img, roi_corners['bottom_right'], zoom_corners['top_left'], line_color, line_thickness)
    img = cv2.resize(img, resize_size, interpolation=cv2.INTER_LANCZOS4)
    cv2.imwrite(output_path, img)
    print(f"Image saved to {output_path}")

In [101]:
def concatenate_images_simple(images, padding=1, bg_color=(0, 0, 0)):
    """
    Simple horizontal concatenation without labels
    
    Args:
        images: list of np.ndarray (H,W,C) or single array (N,H,W,C)
        padding: padding between images in pixels
        bg_color: background color for padding (BGR)
    
    Returns:
        concatenated image
    """
    # Convert to list if input is numpy array (N,H,W,C)
    # if isinstance(images, np.ndarray) and len(images.shape) == 4:
    images = [np.array(cv2.imread(images[i],1)) for i in range(len(images))]
    
    if padding > 0:
        h, w, c = images[0].shape
        padding_strip = np.full((h, padding, c), bg_color, dtype=np.uint8)
        
        result = []
        for i, img in enumerate(images):
            result.append(img)
            if i < len(images) - 1:
                result.append(padding_strip)
        
        return np.hstack(result)
    else:
        return np.hstack(images)


In [102]:
def run_inference_and_save(image_path_to_predict,model_names,suffix_model='',suffix_rs='drive',save_folder='./results/'):
    os.makedirs('./results', exist_ok=True)
    os.makedirs(f'./results/{suffix_rs}', exist_ok=True)
    for model_name in model_names:
        from load_model import load_model_class
        load_model_class(model_name)
        model = torch.load(
            f'/workspace/Retinal-vessels-segmentation/checkpoints/{model_name}{suffix_model}.pt',
            map_location='cuda' if torch.cuda.is_available() else 'cpu',
            weights_only=False
        )
        image_name =image_path_to_predict.split('/')[-1].split('.')[0]
        infer_model(
            model,
            image_path_to_predict,
            output_path=os.path.join(save_folder, f'{suffix_rs}/{model_name}_{image_name}_{suffix_rs}.png'),
            preprocessing_func=preprocessing_img if 'our_net' not in model_name else lambda x: cv2.cvtColor(x, cv2.COLOR_RGB2GRAY)
        )

In [103]:
def save_png_to_new_path(input_path, root_output_path,suf=''):
    img_name = input_path.split('/')[-1].split('.')[0]
    output_path = os.path.join(root_output_path, f'{img_name}_{suf}.png')
    img = cv2.imread(input_path,1)
    cv2.imwrite(output_path, img)

In [104]:
model_names= ['dysta_net','edae_net','fr_net','gtdla','our_net','sfit_net','unet']

In [105]:
suffix_rs = 'drive'
suffix_model = ''

In [106]:
image_path_to_predict = '/workspace/Retinal-vessels-segmentation/data/DRIVE/test/images/02_test.tif'
gt_path_corr = '/workspace/Retinal-vessels-segmentation/data/DRIVE/test/mask/02_manual1.gif'

In [107]:
run_inference_and_save(image_path_to_predict,model_names,suffix_model,suffix_rs)

/venv/main/lib/python3.12/site-packages/torch/nn/modules/module.py:1776: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


In [108]:
save_png_to_new_path(image_path_to_predict, f'/workspace/Retinal-vessels-segmentation/notebooks/Duy/results/{suffix_rs}',suf='origin')
save_png_to_new_path(gt_path_corr, f'/workspace/Retinal-vessels-segmentation/notebooks/Duy/results/{suffix_rs}',suf='gt')

In [109]:
data_dir = f'/workspace/Retinal-vessels-segmentation/notebooks/Duy/results/{suffix_rs}'

In [110]:
pos = (113, 276, 27, 64)

In [111]:
for img_name in os.listdir(data_dir):
    if os.path.isdir(os.path.join(data_dir, img_name)):
        continue
    img_path = os.path.join(data_dir, img_name)
    os.makedirs(os.path.join(data_dir, 'zoomed_insets'), exist_ok=True)
    create_zoom_inset(img_path,os.path.join(data_dir, 'zoomed_insets', 'test_'+img_name),pos=pos,scale_roi=3.55,line_style='dashed')

Image saved to /workspace/Retinal-vessels-segmentation/notebooks/Duy/results/drive/zoomed_insets/test_dysta_net_02_test_drive.png
Image saved to /workspace/Retinal-vessels-segmentation/notebooks/Duy/results/drive/zoomed_insets/test_edae_net_02_test_drive.png
Image saved to /workspace/Retinal-vessels-segmentation/notebooks/Duy/results/drive/zoomed_insets/test_fr_net_02_test_drive.png
Image saved to /workspace/Retinal-vessels-segmentation/notebooks/Duy/results/drive/zoomed_insets/test_gtdla_02_test_drive.png
Image saved to /workspace/Retinal-vessels-segmentation/notebooks/Duy/results/drive/zoomed_insets/test_our_net_02_test_drive.png
Image saved to /workspace/Retinal-vessels-segmentation/notebooks/Duy/results/drive/zoomed_insets/test_sfit_net_02_test_drive.png
Image saved to /workspace/Retinal-vessels-segmentation/notebooks/Duy/results/drive/zoomed_insets/test_unet_02_test_drive.png
Image saved to /workspace/Retinal-vessels-segmentation/notebooks/Duy/results/drive/zoomed_insets/test_02_t

In [112]:
# create_zoom_inset('/workspace/Retinal-vessels-segmentation/notebooks/Duy/results/drive/fr_net_06_test_drive.png','./test.png',scale_roi=4.8,line_style='dashed')

In [113]:
def swap_loc(imgs):
    def filter_key(path):
        if '_gt.' in path:
            return False
        elif '_origin.' in path:
            return False
        elif 'our_net' in path:
            return False
        else :
            pass
        return True
    tmp = [0]*3
    for path in imgs:
        if '_gt.' in path:
            tmp[1] = path
        elif '_origin.' in path:
            tmp[0] = path
        elif 'our_net' in path:
            tmp[2] = path
        else : pass
    return tmp+list(filter(filter_key, imgs))


In [114]:
import shutil
import glob
import os

In [115]:
result3=concatenate_images_simple(swap_loc(glob.glob(f'/workspace/Retinal-vessels-segmentation/notebooks/Duy/results/{suffix_rs}/zoomed_insets/*.png')),0)
os.makedirs(f"/workspace/Retinal-vessels-segmentation/notebooks/Duy/results/{suffix_rs}/rs", exist_ok=True)
cv2.imwrite(f"/workspace/Retinal-vessels-segmentation/notebooks/Duy/results/{suffix_rs}/rs/concatenated_simple_0.png", result3)
Image.open(f"/workspace/Retinal-vessels-segmentation/notebooks/Duy/results/{suffix_rs}/rs/concatenated_simple_0.png").save(
    f"/workspace/Retinal-vessels-segmentation/notebooks/Duy/results/{suffix_rs}/rs/concatenated_simple_0.png",
    dpi=(300, 300)
)

In [116]:
swap_loc(glob.glob(f'/workspace/Retinal-vessels-segmentation/notebooks/Duy/results/{suffix_rs}/zoomed_insets/*.png'))

['/workspace/Retinal-vessels-segmentation/notebooks/Duy/results/drive/zoomed_insets/test_02_test_origin.png',
 '/workspace/Retinal-vessels-segmentation/notebooks/Duy/results/drive/zoomed_insets/test_02_manual1_gt.png',
 '/workspace/Retinal-vessels-segmentation/notebooks/Duy/results/drive/zoomed_insets/test_our_net_02_test_drive.png',
 '/workspace/Retinal-vessels-segmentation/notebooks/Duy/results/drive/zoomed_insets/test_dysta_net_02_test_drive.png',
 '/workspace/Retinal-vessels-segmentation/notebooks/Duy/results/drive/zoomed_insets/test_edae_net_02_test_drive.png',
 '/workspace/Retinal-vessels-segmentation/notebooks/Duy/results/drive/zoomed_insets/test_fr_net_02_test_drive.png',
 '/workspace/Retinal-vessels-segmentation/notebooks/Duy/results/drive/zoomed_insets/test_gtdla_02_test_drive.png',
 '/workspace/Retinal-vessels-segmentation/notebooks/Duy/results/drive/zoomed_insets/test_sfit_net_02_test_drive.png',
 '/workspace/Retinal-vessels-segmentation/notebooks/Duy/results/drive/zoomed_i

In [117]:
import shutil
import glob
import os
shutil.rmtree(f'/workspace/Retinal-vessels-segmentation/notebooks/Duy/results/{suffix_rs}/zoomed_insets')
for png_file in glob.glob(os.path.join(f'/workspace/Retinal-vessels-segmentation/notebooks/Duy/results/{suffix_rs}', '*.png')):
    os.remove(png_file)